# Notebook para la predicción de una métrica cuantitativa en función de los modelos generalizados previamente

## Importación de librerías necesarias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

In [3]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('predict_flow')
spark = spark_utils.spark

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aa571d9c-6a68-4467-b96a-89b4e6fcd140;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 103ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

## Importar información de referencia

In [4]:
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items', 'bronze'))

In [5]:
ASIN = "1610121147"

## Recibir entrada de usuario separada por componente

In [6]:
TITLE = 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black) (OLD MODEL)'
DESCRIPTION = ['Product Description', 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black)', 'From the Manufacturer', 'Canon looked to the very first Elph for inspiration when designing the PowerShot SD1000 Digital Elph, and came up with a quintessential iteration of the icon: slim, clean-lined and fully flat. Inside, the SD1000 Digital Elph looks only to the future: 7.1 megapixels, a 3x optical zoom and advanced DIGIC III ensure top-quality images, while focus is fast and sharp and red-eye is automatically corrected. The large and more colorful LCD screen now has a tough, anti-reflective coating that makes it as durable as it is beautiful.', 'PowerShot SD1000 Highlights', '1x zoom/3x zoom', 'Slim, stylish 7.1-megapixel digital Elph with 3x optical zoom', "Great design is just part of the PowerShot SD1000 Digital Elph story. Inside is all the power you need to capture the moments of your life - beautifully.   The 7.1-megapixel CCD records a wealth of detail - enough to let you enlarge and crop at will. Images are rich and sharp with lifelike depth. The camera's Genuine Canon 3x optical zoom not only gets you in close, but performs with all the clarity and brilliance you'd expect from the world's leader in advanced optics technology.", 'DIGIC III image processor with improved Face Detection and Red-eye Correction', "With DIGIC III, you get images of superior quality, the camera functions at top efficiency and battery life is significantly enhanced. What's more, DIGIC III enables Canon's newly improved Face Detection Technology and Red-eye Correction to give you better, more true-to-life people shots. Simply press the Shutter Button halfway down, and the PowerShot SD1000 Digital Elph automatically pinpoints the faces in the scene and chooses the ideal focus point. To keep every face looking bright and natural - without scary red eyes - the camera controls exposure settings and flash, so every shot is just what you were shooting for.", 'Face Detection AF/AE', "finds multiple faces in the frame and sets the most suitable focus point, when the shutter button is pressed halfway. And an additional feature, Face Detection FE adjusts the flash, based on a person's face on the screen. Exposure and flash are controlled to ensure proper illumination of both the faces and the overall scene, eliminating the common problem of darkened or overexposed faces.", 'Face Detection in action', 'Red-eye Correction', 'detects and automatically corrects red-eye during playback for both regular and flash photography. In unusual cases where red-eye is not automatically detected, it can easily be corrected manually during playback mode from the LCD screen.', 'iSAPS Technology', 'is an entirely original scene-recognition technology developed for digital cameras by Canon. Using an internal database of thousands of different photos, iSAPS works with the fast DIGIC III Image Processor to improve focus speed and accuracy, as well as exposure and white balance.', 'Vivid, high-resolution 2.5-inch PureColor LCD', "The camera's 2.5-inch LCD screen gives you the big picture, whether you're shooting, reviewing or showing off your images. This extra-durable, high-resolution screen with tough scratch-resistant coating on the anti-reflective, PureColor LCD screen offers a crisp, clear picture to make shooting, playback and using the camera's menu functions especially convenient. Clear and bright, it also features Night Display for easy viewing in low light.", 'ISO 1600 and High ISO Auto to reduce image blur and expand low-light shooting capability', 'The PowerShot SD1000 Digital Elph features new ISO 1600 and High ISO Auto settings that reduce the effects of camera shake and sharpen subjects in low-light situations, giving you greater flexibility for shooting.', 'Five movie modes including 30 fps VGA, Time Lapse and Fast Frame Rate', "With a highly flexible movie mode, you can create the movie that's perfect for any application. Select from VGA (640 x 480 pixels) and QVGA (320 x 240 pixels), with frame rates of 30 fps and 15 fps for recording up to 1 hour or 4GB. Also choose from Fast Frame Rate (QVGA; 320 x 240 pixels) recording at 60 fps for up to 1 minute, Compact Movie Mode (QQVGA; 160 x 120 pixels) recording at 15 fps for up to 3 minutes, and Time Lapse (640 x 480) recording at 1 or 2 sec. intervals. The PowerShot SD1000 Digital Elph supports the USB 2.0 Hi-Speed standard, so you'll enjoy the fastest possible data transfer speeds when using a USB 2.0 Hi-Speed compatible computer.", 'Print/Share button for easy direct printing and downloading', "The PowerShot SD1000 Digital Elph's Print/Share button makes direct printing easier than ever. Simply connect the SD1000 Digital Elph to a Canon CP, Selphy or Pixma photo printer or any PictBridge compatible photo printer, press the lighted Print/Share button and print! Also use the Print/Share button to transfer images to a computer (Windows and Macintosh).   Print your own ID photos in 28 different sizes or use the Movie Print function to output multiple stills from a recorded movie on a single sheet with a Canon Selphy compact photo printer.", 'Direct photo printers', 'For desktop large-format printing, try one of the direct photo printers that allow you to print without a computer in one of two ways: plug your compatible PowerShot camera into the direct photo printer using the supplied USB interface cable, or simply insert a memory card into the supplied adapter. You can also connect the printer to your computer for more options. Print high-resolution, borderless images as postcards or 8.5 x 11-inch sheets within minutes.', 'Compact photo printers', "Compact photo printers let you produce versatile, fun 4 x 6-inch postcards, 4 x 8-inch wide greeting cards or credit-card size prints in just two easy steps: connect and press print. Control the printer right from your camera's LCD screen. You get durable, dye-sublimated prints quickly with or without borders. Assorted paper types let you create mini or credit card size labels. You can even take select compact photo printers to a party or an outdoor picnic using an optional rechargeable battery."]
FEATURES = ['7.1-megapixel CCD captures enough detail for photo-quality 15 x 20-inch prints', '3x optical zoom; ISO 1600 and High ISO Auto', 'DIGIC III Image Processor; Face Detection AF/AE', 'Selectable shooting modes and special scene modes', 'Print/Share button makes direct printing simple']

In [7]:
meta_items_title_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_title_text_clean',
    catalog = 'silver.preprocess'
))

In [8]:
import pyspark.sql.functions as F
meta_items_title_text_clean.filter(F.col('parent_asin') == ASIN).limit(1).toPandas()

,parent_asin,title
0,1610121147,NEWEST Black Color Arachnophobia Durable Alumi...


In [9]:
meta_items_title_text_clean.select('parent_asin').distinct().count()

3125022

In [10]:
# TITLE = "Iphone 15 Pro Max"
# DESCRIPTION = [
#     """
#     The iPhone 15 Pro Max represents the pinnacle of Apple's smartphone engineering, combining cutting-edge technology with premium design. This flagship device features a stunning 6.7-inch Super Retina XDR display that delivers exceptional brightness, contrast, and color accuracy. The device is powered by the revolutionary A17 Pro chip, which provides unprecedented performance for gaming, photography, and productivity tasks. With its titanium construction, the iPhone 15 Pro Max offers a perfect balance of durability and lightweight elegance, making it comfortable to hold despite its large screen size.
#     """,
#     """
#     Photography enthusiasts will be amazed by the advanced camera system on the iPhone 15 Pro Max. The device boasts a triple-lens camera array featuring a 48MP main sensor, a 12MP ultra-wide lens, and a 12MP telephoto lens with 5x optical zoom. The new ProRAW and ProRes video recording capabilities allow professional-grade content creation directly from your smartphone. The improved Night mode and Photographic Styles ensure stunning results in any lighting condition, while the Action Button provides quick access to camera functions for capturing those perfect moments.
#     """,
#     """
#     Performance is where the iPhone 15 Pro Max truly shines, thanks to the A17 Pro chip built on an advanced 3-nanometer process. This powerhouse processor delivers up to 20% faster CPU performance and 30% faster GPU performance compared to previous generations. Whether you're editing 4K videos, playing graphics-intensive games, or running multiple apps simultaneously, the device handles everything with remarkable speed and efficiency. The enhanced Neural Engine enables advanced machine learning features and improved battery optimization.
#     """,
#     """
#     Connectivity and battery life have been significantly improved in the iPhone 15 Pro Max. The device supports USB-C connectivity for faster data transfer and charging, while maintaining compatibility with a wide range of accessories. The battery life has been optimized to provide all-day usage, even with heavy multitasking and media consumption. The device also features advanced 5G capabilities, Wi-Fi 6E support, and improved Bluetooth connectivity for seamless integration with other Apple devices and accessories.
#     """,
#     """
#     The iPhone 15 Pro Max runs on iOS 17, which introduces new features like StandBy mode, improved Siri capabilities, and enhanced privacy controls. The device includes Face ID for secure authentication and supports Apple Pay for convenient contactless payments. With storage options ranging from 256GB to 1TB, users can choose the capacity that best suits their needs. The device is also water and dust resistant with an IP68 rating, ensuring protection against accidental spills and exposure to the elements.
#     """,
# ]
# FEATURES = [
#     """
#     The device features a 6.7-inch Super Retina XDR display with ProMotion technology supporting refresh rates up to 120Hz for incredibly smooth scrolling and interactions.
#     """,
#     """
#     Powered by the A17 Pro chip with a 6-core CPU and 6-core GPU, delivering exceptional performance for demanding applications and games.
#     """,
#     """
#     Equipped with a professional-grade triple-camera system including a 48MP main camera, 12MP ultra-wide, and 12MP telephoto with 5x optical zoom capability.
#     """,
#     """
#     Constructed from aerospace-grade titanium, making it both incredibly durable and surprisingly lightweight for a device of this size.
#     """,
#     """
#     Features USB-C connectivity for faster charging and data transfer, along with all-day battery life optimized for intensive use.
#     """,
# ]

In [11]:
from src.utils.spark import SparkUtils
from src.utils.preprocessors.clean_words import CleanWords
from src.utils.models.clustering.LSHNeighborsClustering import LSHNeighborsClustering
from src.utils.models.encoding.SummarizeEncoding import SummarizeEncoding
from src.utils.models.encoding.SentenceEncoder import SentenceEncoder
from src.gold.training.pca import PCAEncoder

In [12]:
from src.utils.testers.FullTester import FullTester
full_tester = FullTester(
    spark=spark,
    spark_utils=spark_utils,
    use_mini_llm=False
)

full_tester.set_components({
    "title": TITLE,
    "description": DESCRIPTION,
    "features": FEATURES,
})

full_tester.clean_components()
full_tester.separate_sentences_per_component()
full_tester.encode_sentences()
full_tester.summarize_sentences_by_component()
full_tester.pca_encode()
full_tester.find_pairs()
# full_tester.build_models_inputs()
full_tester.generate_prediction()

I0000 00:00:1762927430.735521 1148028 service.cc:146] XLA service 0x398fe2c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762927430.735586 1148028 service.cc:154]   StreamExecutor device (0): Host, Default Version


01:04:14.762 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


01:04:19.578 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


01:04:22.593 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


...

01:05:17.174 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


01:05:40.799 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


01:05:43.726 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
full_tester.jtbdbased.df_reviews_relevance.count()

337